In [ ]:
import numpy as np
import pandas
from sklearn.decomposition import PCA
from sklearn import preprocessing
from tqdm import tqdm


def predict(df):

    res = df.copy()
    for categorical_col in res.select_dtypes(include=['object']).columns:
        le = preprocessing.LabelEncoder()
        res[categorical_col] = le.fit_transform(res[categorical_col])
    model = PCA(n_components=2, random_state=0)
    W = model.fit_transform(res.fillna(0))
    H = model.components_
    R = np.dot(W,H)
    return R


def extract_value(row):
    if row["type"] == "num":
        return preds[row["cell_id_df"]][int(row["cell_id_y"]), dfs[row["cell_id_df"]].columns.get_loc(row["cell_id_x"])]
    else:
        return dfs[row["cell_id_df"]][row["cell_id_x"]].value_counts().index[0]


sub = pandas.read_csv("../input/gapsingaps/sample_submission.csv")
sub["cell_id_split"] = sub["cell_id"].str.split(",")
sub["cell_id_df"] = [d[0] for d in sub["cell_id_split"]]
sub["cell_id_x"] = [d[1] for d in sub["cell_id_split"]]
sub["cell_id_y"] = [d[2] for d in sub["cell_id_split"]]

dfs = {
    "D1": pandas.read_csv("../input/gapsingaps/data/D1.csv"),
    "D2": pandas.read_csv("../input/gapsingaps/data/D2.csv"),
    "D3": pandas.read_csv("../input/gapsingaps/data/D3.csv"),
    "D4": pandas.read_csv("../input/gapsingaps/data/D4.csv"),
    "D5": pandas.read_csv("../input/gapsingaps/data/D5.csv"),
    "D6": pandas.read_csv("../input/gapsingaps/data/D6.csv"),
    "D7": pandas.read_csv("../input/gapsingaps/data/D7.csv"),
    "D8": pandas.read_csv("../input/gapsingaps/data/D8.csv"),
    "D9": pandas.read_csv("../input/gapsingaps/data/D9.csv"),
    "D10": pandas.read_csv("../input/gapsingaps/data/D10.csv")   
}

preds = {d: predict(dfs[f"{d}"]) for d in dfs.keys()}

tqdm.pandas()
sub["value"] = sub.progress_apply(extract_value, axis=1)
sub[["cell_id", "value", "type"]].to_csv("submission.csv", index=False)